In [2]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
TEST_SIZE = 0.2

RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

In [3]:
ideb = pd.read_csv(
    RAW_PATH + "ideb_2.csv",
    sep=";",
    encoding="utf-8",
    dtype={"CO_MUNICIPIO": "string"},
    na_values=["-"]
)

ideb2023 = ideb.loc[
    ideb["REDE"] == "Pública",
    [
        "SG_UF",
        "CO_MUNICIPIO",
        "NO_MUNICIPIO",
        "VL_OBSERVADO_2023"
    ]
].copy()

ideb2023 = ideb2023.rename(columns={
    "SG_UF": "UF",
    "CO_MUNICIPIO": "COD",
    "NO_MUNICIPIO": "NAME",
    "VL_OBSERVADO_2023": "IDEB"
})

ideb2023["IDEB"] = pd.to_numeric(
    ideb2023["IDEB"],
    errors="coerce"
)

ideb2023 = ideb2023.dropna(subset=["IDEB"])

display(ideb2023.head())

,UF,COD,NAME,IDEB
2,RO,1100015,Alta Floresta D'Oeste,5.3
5,RO,1100023,Ariquemes,5.4
8,RO,1100031,Cabixi,5.8
11,RO,1100049,Cacoal,5.7
14,RO,1100056,Cerejeiras,5.8


In [4]:
escolasRaw = pd.read_csv(
    RAW_PATH + "microdados_ed_basica_2022.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)

escolas = escolasRaw.loc[
    (escolasRaw["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
    (escolasRaw["TP_DEPENDENCIA"].isin([1, 2, 3])) &
    (escolasRaw["IN_FUND_AI"] == 1),
    [
        "CO_MUNICIPIO",
        "CO_ENTIDADE",
        "IN_INTERNET",
        "QT_MAT_FUND_AI",
        "QT_DOC_FUND_AI"
    ]
].copy()

escolas = escolas.rename(columns={
    "CO_MUNICIPIO": "COD",
    "CO_ENTIDADE": "COD_ESCOLA",
    "IN_INTERNET": "ESCOLAS_COM_INTERNET",
    "QT_MAT_FUND_AI": "MATRICULAS",
    "QT_DOC_FUND_AI": "DOCENTES"
})

escolas["COD"] = (
    escolas["COD"].astype("Int64").astype("string")
)

municipios = escolas.groupby("COD").agg(
    COD_ESCOLA=("COD_ESCOLA", "nunique"),
    ESCOLAS_COM_INTERNET=("ESCOLAS_COM_INTERNET", "mean"),
    MATRICULAS=("MATRICULAS", "sum"),
    DOCENTES=("DOCENTES", "sum")
).reset_index()

municipios["ALUNOS_POR_DOCENTE"] = (
    municipios["MATRICULAS"] /
    municipios["DOCENTES"].replace(0, np.nan)
)

municipios["ESCOLAS_COM_INTERNET"] *= 100

display(municipios.head())


,COD,COD_ESCOLA,ESCOLAS_COM_INTERNET,MATRICULAS,DOCENTES,ALUNOS_POR_DOCENTE
0,1100015,23,78.26087,1563.0,86.0,18.174419
1,1100023,23,100.00000,6183.0,273.0,22.648352
2,1100031,3,100.00000,358.0,28.0,12.785714
3,1100049,33,100.00000,4372.0,232.0,18.844828
4,1100056,5,100.00000,1058.0,55.0,19.236364


In [6]:
urbanizacaoRaw = pd.read_csv(
    RAW_PATH + "tabela9923.csv",
    sep=";",
    encoding="utf-8"
)

urbanizacao = urbanizacaoRaw[
    ["Cód.", "UrbanaPorcentagem"]
].copy()

urbanizacao = urbanizacao.rename(columns={
    "Cód.": "COD",
    "UrbanaPorcentagem": "TAXA_URBANIZACAO"
})

urbanizacao["TAXA_URBANIZACAO"] = (
    urbanizacao["TAXA_URBANIZACAO"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

display(urbanizacao.head())

,COD,TAXA_URBANIZACAO
0,1100015,60.35
1,1100023,86.70
2,1100031,53.19
3,1100049,82.76
4,1100056,88.84


In [8]:
def NormalizeCodCity(column):
    return (
        column.astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
        .str.zfill(7)
    )

In [9]:
ideb2023["COD"] = NormalizeCodCity(ideb2023["COD"])
municipios["COD"] = NormalizeCodCity(municipios["COD"])
urbanizacao["COD"] = NormalizeCodCity(urbanizacao["COD"])

In [10]:
dataset = ideb2023.merge(
    municipios,
    on="COD",
    how="left",
    validate="one_to_one"
)

dataset = dataset.merge(
    urbanizacao,
    on="COD",
    how="left",
    validate="one_to_one"
)

# dataset = dataset.merge(
#     population,
#     on="COD",
#     how="left",
#     validate="one_to_one"
# )

# dataset = dataset.merge(
#     wealth,
#     on="COD",
#     how="left",
#     validate="one_to_one"
# )

In [11]:
print("Dimensões: ", dataset.shape)

print("\nValores Ausentes: ")
print(dataset.isna().sum())

print("\nTipos: ")
print(dataset.dtypes)

print("\nDuplicados: ")
print(dataset["COD"].duplicated().sum())

display(dataset.head())

dataset.to_csv(
    PROCESSED_PATH + "dataset_final.csv",
    sep=";",
    index=False,
    encoding="utf-8"
)

Dimensões:  (5430, 10)

Valores Ausentes: 
UF                      0
COD                     0
NAME                    0
IDEB                    0
COD_ESCOLA              0
ESCOLAS_COM_INTERNET    0
MATRICULAS              0
DOCENTES                0
ALUNOS_POR_DOCENTE      0
TAXA_URBANIZACAO        0
dtype: int64

Tipos: 
UF                          str
COD                      string
NAME                        str
IDEB                    float64
COD_ESCOLA                int64
ESCOLAS_COM_INTERNET    float64
MATRICULAS              float64
DOCENTES                float64
ALUNOS_POR_DOCENTE      float64
TAXA_URBANIZACAO        float64
dtype: object

Duplicados: 
0


,UF,COD,NAME,IDEB,COD_ESCOLA,ESCOLAS_COM_INTERNET,MATRICULAS,DOCENTES,ALUNOS_POR_DOCENTE,TAXA_URBANIZACAO
0,RO,1100015,Alta Floresta D'Oeste,5.3,23,78.26087,1563.0,86.0,18.174419,60.35
1,RO,1100023,Ariquemes,5.4,23,100.00000,6183.0,273.0,22.648352,86.70
2,RO,1100031,Cabixi,5.8,3,100.00000,358.0,28.0,12.785714,53.19
3,RO,1100049,Cacoal,5.7,33,100.00000,4372.0,232.0,18.844828,82.76
4,RO,1100056,Cerejeiras,5.8,5,100.00000,1058.0,55.0,19.236364,88.84
